In [1]:
import warnings
from pathlib import Path
warnings.filterwarnings('ignore')

import pandas as pd
import numpy as np
import joblib

# add path to project root
import sys
PROJECT_ROOT = Path.cwd()

if "notebooks" in PROJECT_ROOT.parts:
    PROJECT_ROOT = PROJECT_ROOT.parent
    
sys.path.append(str(PROJECT_ROOT))


from src.preprocess.gaming_text_preprocessor import GamingTextPreprocessor
from src.preprocess.gaming_text_labeler import GamingTextLabeler
from src.ensemble.ensemble import Ensemble
from src.model.bert_collection import BertToxicityModelCollection
from src.ensemble.score import ClassificationMetrics

# instantiate preprocessor and labeler
gaming_labeler = GamingTextLabeler()
metrics = ClassificationMetrics()



In [2]:
CONFIG = {
    'seed': 7524,
    'data_dir': PROJECT_ROOT / 'data' / 'processed_data',
    'model_dir': PROJECT_ROOT / 'models',
    'text_col': 'message',
    'label_col': 'label'
}

np.random.seed(CONFIG['seed'])
seed = CONFIG['seed']
tc, lc = CONFIG['text_col'], CONFIG['label_col']

print('CONFIG loaded. Text column:', tc)

CONFIG loaded. Text column: message


In [3]:
# load WOT train data
d = CONFIG['data_dir']

wot_train = pd.read_parquet(d / 'wot' / 'x_train.parquet')
wot_train["data_source"] = "wot"

# load DOTA training data
dota_train = pd.read_parquet(d / 'dota' / 'x_train.parquet')
dota_train["data_source"] = "dota"

# combine datasets
train = pd.concat([wot_train, dota_train], ignore_index=True)

# clean labels and create 3-class label column
train = gaming_labeler.convert_three_class(
    train, 
    label_column=lc, 
    data_source_column='data_source',
    output_column='label_3class'
)

In [4]:
wot_val = pd.read_parquet(d / 'wot' / 'x_validation.parquet')
wot_val["data_source"] = "wot"
dota_val = pd.read_parquet(d / 'dota' / 'x_validation.parquet')
dota_val["data_source"] = "dota"

# combine holdout datasets
val = pd.concat([wot_val, dota_val], ignore_index=True)

# clean labels and create 3-class label column
val = gaming_labeler.convert_three_class(
    val, 
    label_column=lc, 
    data_source_column='data_source',
    output_column='label_3class'
)

# Create Objects 

In [5]:
bert_multi_model = BertToxicityModelCollection(
    model_names=["dota_multi", "jigsaw_multi", "wot_multi"],
)

Loading dota_multi from jforward/bert-toxicity/dota_multi_model...


tokenizer_config.json:   0%|          | 0.00/351 [00:00<?, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

config.json:   0%|          | 0.00/969 [00:00<?, ?B/s]

dota_multi_model/model.safetensors:   0%|          | 0.00/438M [00:00<?, ?B/s]

...

Loading wot_multi from jforward/bert-toxicity/wot_multi_model...


tokenizer_config.json:   0%|          | 0.00/351 [00:00<?, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

config.json:   0%|          | 0.00/969 [00:00<?, ?B/s]

wot_multi_model/model.safetensors:   0%|          | 0.00/438M [00:00<?, ?B/s]

In [6]:
bert_multi_ensemble = Ensemble(
    model_collections=[bert_multi_model]
)

# Simple Majority Rules

In [7]:
pred_train = bert_multi_ensemble.predict_simple_majority(train[tc])
metrics.metrics(train['label_3class'], pred_train)

Predicting with SimpleMajority...


{'CV Macro F1': 0.5327168007618174,
 'CV Weighted F1': 0.7743964328367929,
 'Accuracy': 0.8191669614761576,
 'Coverage': 1.0,
 'Precision': 0.7113251507832531}

In [8]:
pred_train = bert_multi_ensemble.predict_simple_majority(val[tc])
metrics.metrics(val['label_3class'], pred_train)

Predicting with SimpleMajority...


{'CV Macro F1': 0.4990977903561413,
 'CV Weighted F1': 0.7724849682856554,
 'Accuracy': 0.8178353658536586,
 'Coverage': 1.0,
 'Precision': 0.6380795925531287}